# 🦺 PPE Kit Detection — Construction Site Safety (Google Colab)

This notebook trains a **YOLO11** object detection model to monitor PPE compliance on construction sites.  
It is fully optimised for **Google Colab T4 / A100 GPU** — expect ~15–25 min training for 50 epochs.

---

## Dataset — 11 Classes

| Class ID | Label | Type |
|----------|-------|------|
| 0 | Helmet | ✅ Positive |
| 1 | Gloves | ✅ Positive |
| 2 | Vest | ✅ Positive |
| 3 | Boots | ✅ Positive |
| 4 | Goggles | ✅ Positive |
| 5 | none | ⬜ Neutral |
| 6 | Person | ✅ Positive |
| 7 | no_helmet | ❌ Negative |
| 8 | no_goggle | ❌ Negative |
| 9 | no_gloves | ❌ Negative |
| 10 | no_boots | ❌ Negative |

**Dataset Source:** [Kaggle — PPE Kit Detection (Construction Site Workers)](https://www.kaggle.com/datasets/ketakichalke/ppe-kit-detection-construction-site-workers)

---

### ⚡ Recommended Runtime
Go to **Runtime → Change runtime type → T4 GPU** before running.

## 📦 Step 1 — Install Required Packages

In [ ]:
# Install all required packages
!pip install ultralytics imagehash albumentations gradio kagglehub --quiet

import subprocess, sys

# Verify GPU is available
import torch
print(f"✅ PyTorch  : {torch.__version__}")
print(f"✅ CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU Name : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected — go to Runtime → Change runtime type → T4 GPU")

import ultralytics
ultralytics.checks()
print("✅ All packages ready!")

## ☁️ Step 2 — Mount Google Drive & Configure Paths

Mounting Drive lets you **save the trained model persistently** — it won't disappear when the Colab session ends.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os

# ── All outputs are written under /content/ppe_detection/  ───────────────────
# The trained model is ALSO copied to Google Drive so it survives session resets.

DRIVE_BASE   = "/content/drive/MyDrive/PPE_Detection"   # ← change folder if needed
CONTENT_BASE = "/content/ppe_detection"

os.makedirs(DRIVE_BASE,   exist_ok=True)
os.makedirs(CONTENT_BASE, exist_ok=True)

print(f"📁 Working dir  : {CONTENT_BASE}")
print(f"💾 Drive save   : {DRIVE_BASE}")
print("✅ Paths configured.")

## 📥 Step 3 — Download Dataset from Kaggle

Using **`kagglehub`** — the official Kaggle Python client. It automatically handles authentication via `KAGGLE_API_TOKEN` and caches the dataset locally so re-runs are instant.

> ⚠️ **Security tip:** Store your token in **Colab Secrets** (`🔑` left sidebar, key = `KAGGLE_API_TOKEN`) and uncomment Option B below to avoid hardcoding credentials.

In [ ]:
import os

# ── Option A: hardcoded token (quick start) ───────────────────────────────────
KAGGLE_API_TOKEN = "KGAT_1f4096fef9cc880e5642ac3079fc1353"

# ── Option B: Colab Secrets (recommended — keeps token out of the notebook) ──
# from google.colab import userdata
# KAGGLE_API_TOKEN = userdata.get("KAGGLE_API_TOKEN")

# kagglehub reads this env-var automatically
os.environ["KAGGLE_API_TOKEN"] = KAGGLE_API_TOKEN
print(f"✅ KAGGLE_API_TOKEN set ({KAGGLE_API_TOKEN[:12]}...)")

In [ ]:
import kagglehub

# ── Download latest version via kagglehub ─────────────────────────────────────
# First run: downloads & caches under ~/.cache/kagglehub/
# Subsequent runs: returns the cached path instantly (no re-download)
print("⬇️  Downloading dataset via kagglehub...")
path = kagglehub.dataset_download("ketakichalke/ppe-kit-detection-construction-site-workers")
print(f"\n📂 Path to dataset files: {path}")

# kagglehub extracts to a versioned cache dir — point RAW_DATA_PATH at it
RAW_DATA_PATH = path   # e.g. ~/.cache/kagglehub/datasets/ketakichalke/.../data

# ── Show what was downloaded ──────────────────────────────────────────────────
for root, dirs, files_list in os.walk(RAW_DATA_PATH):
    level = root.replace(RAW_DATA_PATH, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root) or 'dataset'}/")
    if level < 2:
        for f in files_list[:5]:
            print(f"{indent}  {f}")
        if len(files_list) > 5:
            print(f"{indent}  ... ({len(files_list) - 5} more files)")

print("\n✅ Dataset ready!")

## 🔧 Step 4 — Imports & Central Configuration

In [ ]:
# ── System / File handling ──────────────────────────────────────────────────
import os, glob, random, shutil, json, time, logging, warnings
from collections import Counter, defaultdict
from pathlib import Path

warnings.filterwarnings("ignore")

# ── Data manipulation & visualization ───────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from scipy.spatial.distance import jensenshannon

# ── Image processing ─────────────────────────────────────────────────────────
import cv2
from PIL import Image
import imagehash
import albumentations as A

# ── YOLO ─────────────────────────────────────────────────────────────────────
from ultralytics import YOLO
import yaml

print("✅ All libraries loaded successfully!")

In [ ]:
class CFG:
    # ── Debug / Reproducibility ────────────────────────────────────────────
    DEBUG   = False          # set True for a quick 3-epoch sanity check
    SEED    = 42

    # ── Dataset classes ────────────────────────────────────────────────────
    PPE_CLASSES = {
        0: "Helmet",   1: "Gloves",    2: "Vest",
        3: "Boots",    4: "Goggles",   5: "none",
        6: "Person",   7: "no_helmet", 8: "no_goggle",
        9: "no_gloves", 10: "no_boots",
    }
    NUM_CLASSES = len(PPE_CLASSES)

    POSITIVE_CLASSES = {"Helmet", "Gloves", "Vest", "Boots", "Goggles", "Person"}
    NEGATIVE_CLASSES = {"no_helmet", "no_goggle", "no_gloves", "no_boots"}

    # ── Split ratios ───────────────────────────────────────────────────────
    TARGET_RATIOS = {"train": 0.75, "valid": 0.15, "test": 0.10}

    # ── Training hyper-parameters ──────────────────────────────────────────
    #   yolo11n → fastest (edge/demo)
    #   yolo11s → recommended for Colab T4  ← default here
    #   yolo11m → best accuracy (needs A100)
    EPOCHS     = 3  if DEBUG else 50
    PATIENCE   = 5  if DEBUG else 20
    BATCH_SIZE = 8  if DEBUG else 32       # T4 can handle 32 at 640px with yolo11s
    IMGSZ      = 640
    FRACTION   = 0.05 if DEBUG else 1.0
    BASE_MODEL = "yolo11s.pt"              # upgrade from 11n → better mAP on Colab GPU
    DEVICE     = "0"                       # GPU 0;  use "cpu" if no GPU
    LR0        = 0.01
    COS_LR     = True
    OPTIMIZER  = "AdamW"
    RECT       = False

    # ── Paths ─────────────────────────────────────────────────────────────
    DATA_PATH    = RAW_DATA_PATH           # /content/kaggle_data/data
    WORKING_PATH = CONTENT_BASE            # /content/ppe_detection
    FOLDERS      = ["train", "valid", "test"]

    # Derived paths
    YAML_PATH         = f"{WORKING_PATH}/dataset.yaml"
    TRAIN_RESULTS_CSV = f"{WORKING_PATH}/train_results.csv"
    MODEL_PATH        = f"{WORKING_PATH}/ppe_yolo11s/weights/best.pt"
    OUTPUT_MODEL_PATH = f"{WORKING_PATH}/best_ppe_model.pt"
    METADATA_PATH     = f"{WORKING_PATH}/model_metadata.json"
    DRIVE_MODEL_PATH  = f"{DRIVE_BASE}/best_ppe_model.pt"
    DRIVE_META_PATH   = f"{DRIVE_BASE}/model_metadata.json"

    @staticmethod
    def set_seed():
        random.seed(CFG.SEED)
        np.random.seed(CFG.SEED)
        os.environ["PYTHONHASHSEED"] = str(CFG.SEED)

CFG.set_seed()
print(f"📌 Classes   : {CFG.NUM_CLASSES}")
print(f"📌 Debug     : {CFG.DEBUG}")
print(f"📌 Model     : {CFG.BASE_MODEL}")
print(f"📌 Epochs    : {CFG.EPOCHS}")
print(f"📌 Batch     : {CFG.BATCH_SIZE}")
print(f"📌 Data path : {CFG.DATA_PATH}")
print(f"📌 Work path : {CFG.WORKING_PATH}")

## 🗂️ Step 5 — Folder Setup & Logger

In [ ]:
LOG_PATH = f"{CFG.WORKING_PATH}/training_log.txt"

def setup_logger():
    logger = logging.getLogger("ppe_detection")
    if not logger.handlers:
        logger.setLevel(logging.INFO)
        fh = logging.FileHandler(LOG_PATH)
        fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s",
                                          datefmt="%Y-%m-%d %H:%M:%S"))
        logger.addHandler(fh)
    return logger

logger = setup_logger()

def create_folder_structure():
    for folder in CFG.FOLDERS:
        os.makedirs(os.path.join(CFG.WORKING_PATH, folder, "images"), exist_ok=True)
        os.makedirs(os.path.join(CFG.WORKING_PATH, folder, "labels"), exist_ok=True)
    os.makedirs(os.path.join(CFG.WORKING_PATH, "runs"), exist_ok=True)
    print("✅ Folder structure created")

create_folder_structure()

## 🔍 Step 6 — Exploratory Data Analysis (EDA)

In [ ]:
# ── Discover raw dataset folders ──────────────────────────────────────────────
RAW_FOLDERS = []
for candidate in ["train", "val", "valid", "test"]:
    img_dir = os.path.join(CFG.DATA_PATH, "images", candidate)
    if os.path.exists(img_dir):
        RAW_FOLDERS.append(candidate)

print(f"📂 Found raw split folders: {RAW_FOLDERS}\n")

# ── Count images & labels per split ──────────────────────────────────────────
summary_rows = []
for folder in RAW_FOLDERS:
    img_dir = os.path.join(CFG.DATA_PATH, "images", folder)
    lbl_dir = os.path.join(CFG.DATA_PATH, "labels", folder)
    images  = set(os.path.splitext(f)[0] for f in os.listdir(img_dir)
                  if f.lower().endswith((".jpg", ".png", ".jpeg")))
    labels  = set(os.path.splitext(f)[0] for f in os.listdir(lbl_dir)
                  if f.endswith(".txt"))
    missing_l = images - labels
    missing_i = labels - images
    summary_rows.append({
        "Split": folder, "Images": len(images), "Labels": len(labels),
        "Missing Labels": len(missing_l), "Missing Images": len(missing_i)
    })
    print(f"📂 {folder.upper():6s}  images={len(images):5d}  labels={len(labels):5d}  "
          f"missing_labels={len(missing_l)}  missing_images={len(missing_i)}")

df_inventory = pd.DataFrame(summary_rows)
print("\n", df_inventory.to_string(index=False))

In [ ]:
def build_master_df(data_path, raw_folders, binary=True):
    records = []
    for folder in raw_folders:
        lbl_dir = os.path.join(data_path, "labels", folder)
        for lbl_file in glob.glob(os.path.join(lbl_dir, "*.txt")):
            img_name  = os.path.splitext(os.path.basename(lbl_file))[0]
            class_vals = {cls: 0 for cls in CFG.PPE_CLASSES}
            with open(lbl_file) as f:
                for line in f:
                    parts = line.strip().split()
                    if not parts: continue
                    cid = int(parts[0])
                    if cid in CFG.PPE_CLASSES:
                        class_vals[cid] = 1 if binary else class_vals[cid] + 1
            row = {"filename": img_name, "origin": folder}
            row.update(class_vals)
            records.append(row)
    return pd.DataFrame(records)

df_total = build_master_df(CFG.DATA_PATH, RAW_FOLDERS, binary=True)
print(f"📊 Total images: {len(df_total)}")
print(df_total.head())

In [ ]:
# ── Class distribution chart ─────────────────────────────────────────────────
df_count   = build_master_df(CFG.DATA_PATH, RAW_FOLDERS, binary=False)
class_cols = list(CFG.PPE_CLASSES.keys())
total_counts = df_count[class_cols].sum()
class_names  = [CFG.PPE_CLASSES[c] for c in class_cols]

colors = ["#2ECC71" if CFG.PPE_CLASSES[c] in CFG.POSITIVE_CLASSES
          else "#E74C3C" if CFG.PPE_CLASSES[c] in CFG.NEGATIVE_CLASSES
          else "#95A5A6" for c in class_cols]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
axes[0].bar(class_names, total_counts.values, color=colors, edgecolor="black", linewidth=0.5)
axes[0].set_title("Total Annotation Count per Class", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Class"); axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=45)

axes[1].pie(total_counts.values, labels=class_names, colors=colors,
            autopct="%1.1f%%", startangle=140, textprops={"fontsize": 8})
axes[1].set_title("Class Distribution (Proportion)", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig(f"{CFG.WORKING_PATH}/class_distribution.png", dpi=120, bbox_inches="tight")
plt.show()
print("🟢 Positive  |  🔴 Negative (missing PPE)  |  ⚪ Neutral")

In [ ]:
COLOR_MAP = {
    "Helmet": (0, 200, 0),    "Gloves": (0, 150, 255),
    "Vest":   (255, 200, 0),  "Boots":  (0, 255, 255),
    "Goggles":(200, 0, 255),  "Person": (100, 100, 255),
    "none":   (180, 180, 180),
    "no_helmet": (255, 0, 0), "no_goggle": (200, 0, 100),
    "no_gloves": (255, 80, 0),"no_boots": (150, 0, 0),
}

def draw_yolo_boxes(image_path, label_path, class_map=CFG.PPE_CLASSES):
    img = cv2.imread(image_path)
    if img is None: return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5: continue
                cid = int(parts[0])
                xc, yc, bw, bh = map(float, parts[1:5])
                x1, y1 = int((xc - bw/2)*w), int((yc - bh/2)*h)
                x2, y2 = int((xc + bw/2)*w), int((yc + bh/2)*h)
                name  = class_map.get(cid, str(cid))
                color = COLOR_MAP.get(name, (128, 128, 128))
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                cv2.putText(img, name, (x1, max(y1-5, 10)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
    return img

# Show 6 random images with GT boxes
sample_folder = RAW_FOLDERS[0]
img_dir = os.path.join(CFG.DATA_PATH, "images", sample_folder)
lbl_dir = os.path.join(CFG.DATA_PATH, "labels", sample_folder)
all_imgs = [f for f in os.listdir(img_dir) if f.lower().endswith((".jpg",".png",".jpeg"))]
samples  = random.sample(all_imgs, min(6, len(all_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
for ax, fname in zip(axes.flatten(), samples):
    base    = os.path.splitext(fname)[0]
    img_rgb = draw_yolo_boxes(os.path.join(img_dir, fname),
                              os.path.join(lbl_dir, base + ".txt"))
    if img_rgb is not None:
        ax.imshow(img_rgb)
    ax.axis("off"); ax.set_title(fname[:40], fontsize=7)

plt.suptitle(f"Ground-Truth Boxes — '{sample_folder}' split", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CFG.WORKING_PATH}/sample_images.png", dpi=120, bbox_inches="tight")
plt.show()

## 🔀 Step 7 — Stratified Re-split & File Copy

In [ ]:
def stratified_split(df, target_ratios, seed=CFG.SEED):
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    n, n_train = len(df), int(len(df) * target_ratios["train"])
    n_valid = int(n * target_ratios["valid"])
    return (df.iloc[:n_train].copy(),
            df.iloc[n_train:n_train+n_valid].copy(),
            df.iloc[n_train+n_valid:].copy())

df_train, df_valid, df_test = stratified_split(df_total, CFG.TARGET_RATIOS)
total = len(df_train) + len(df_valid) + len(df_test)
print(f"✅ Total  : {total}")
print(f"   Train  : {len(df_train)} ({len(df_train)/total:.1%})")
print(f"   Valid  : {len(df_valid)} ({len(df_valid)/total:.1%})")
print(f"   Test   : {len(df_test)}  ({len(df_test)/total:.1%})")
assert total == len(df_total), "Data loss during split!"
print("✅ No images lost.")

# ── Distribution after split ──────────────────────────────────────────────────
def class_pct(df):
    return {CFG.PPE_CLASSES[c]: df[c].sum() / len(df) * 100 for c in CFG.PPE_CLASSES}

splits = {"Train": class_pct(df_train), "Valid": class_pct(df_valid), "Test": class_pct(df_test)}
df_dist = pd.DataFrame([
    {"Class": cls, "Percentage": pct, "Split": split}
    for split, data in splits.items() for cls, pct in data.items()
])
plt.figure(figsize=(14, 5))
sns.barplot(data=df_dist, x="Class", y="Percentage", hue="Split", palette="Set2")
plt.title("Class Distribution After Re-split", fontsize=14, fontweight="bold")
plt.xticks(rotation=45, ha="right"); plt.tight_layout()
plt.savefig(f"{CFG.WORKING_PATH}/distribution_after_split.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
EXTS = [".jpg", ".jpeg", ".png", ".bmp"]

def move_files(df, subset, raw_data_path=CFG.DATA_PATH):
    dst_img  = os.path.join(CFG.WORKING_PATH, subset, "images")
    dst_lbl  = os.path.join(CFG.WORKING_PATH, subset, "labels")
    counters = {"copied": 0, "missing_img": 0, "missing_lbl": 0}
    for _, row in df.iterrows():
        stem   = Path(row["filename"]).stem
        origin = row["origin"]
        img_src = next(
            (os.path.join(raw_data_path, "images", origin, stem + ext)
             for ext in EXTS
             if os.path.exists(os.path.join(raw_data_path, "images", origin, stem + ext))),
            None
        )
        if img_src is None:
            counters["missing_img"] += 1; continue
        lbl_src = os.path.join(raw_data_path, "labels", origin, stem + ".txt")
        if not os.path.exists(lbl_src):
            counters["missing_lbl"] += 1; continue
        shutil.copy(img_src, os.path.join(dst_img, os.path.basename(img_src)))
        shutil.copy(lbl_src, os.path.join(dst_lbl, stem + ".txt"))
        counters["copied"] += 1
    return counters

for subset, df_sub in [("train", df_train), ("valid", df_valid), ("test", df_test)]:
    s = move_files(df_sub, subset)
    print(f"[{subset:>5}]  copied={s['copied']:4d}  "
          f"missing_img={s['missing_img']}  missing_lbl={s['missing_lbl']}")
print("\n✅ All files copied.")

# ── pHash deduplication ───────────────────────────────────────────────────────
def find_and_remove_duplicates(base_path, hash_thresh=8):
    hash_map = {}; removed = 0
    for subset in CFG.FOLDERS:
        img_dir = os.path.join(base_path, subset, "images")
        lbl_dir = os.path.join(base_path, subset, "labels")
        if not os.path.isdir(img_dir): continue
        for img_path in sorted(glob.glob(os.path.join(img_dir, "*"))):
            try: h = str(imagehash.phash(Image.open(img_path)))
            except Exception: continue
            if h in hash_map:
                os.remove(img_path)
                lbl_path = os.path.join(lbl_dir, Path(img_path).stem + ".txt")
                if os.path.exists(lbl_path): os.remove(lbl_path)
                removed += 1
            else:
                hash_map[h] = img_path
    return removed

n_removed = find_and_remove_duplicates(CFG.WORKING_PATH)
print(f"🗑️  Duplicates removed: {n_removed}")
for subset in CFG.FOLDERS:
    imgs = glob.glob(os.path.join(CFG.WORKING_PATH, subset, "images", "*"))
    lbls = glob.glob(os.path.join(CFG.WORKING_PATH, subset, "labels", "*.txt"))
    print(f"   [{subset:>5}]  images={len(imgs)}  labels={len(lbls)}")

## 🔧 Step 8 — Data Augmentation

Rare negative-PPE classes (`no_goggle`, `no_boots`, `no_helmet`) are synthetically expanded in the **training split only** using Albumentations: horizontal flip, brightness / contrast, HSV shift, and Gaussian noise.

In [ ]:
AUG_PIPELINE = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.7),
        A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=30, val_shift_limit=20, p=0.5),
        A.GaussNoise(var_limit=(10, 50), p=0.3),
        A.CLAHE(clip_limit=2.0, p=0.3),
    ],
    bbox_params=A.BboxParams(format="yolo", label_fields=["class_labels"], min_visibility=0.3),
)

def read_yolo_labels(lbl_path):
    if not os.path.exists(lbl_path): return [], []
    rows = []
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                rows.append([int(parts[0])] + [float(x) for x in parts[1:]])
    if not rows: return [], []
    arr = np.array(rows)
    return arr[:, 0].astype(int).tolist(), arr[:, 1:].tolist()

def augment_image(img_path, lbl_path, out_img_dir, out_lbl_dir, n_copies=2):
    image = cv2.imread(img_path)
    if image is None: return 0
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    classes, bboxes = read_yolo_labels(lbl_path)
    stem, ext, saved = Path(img_path).stem, Path(img_path).suffix, 0
    for i in range(n_copies):
        try: aug = AUG_PIPELINE(image=image, bboxes=bboxes, class_labels=classes)
        except Exception: continue
        out_stem = f"{stem}_aug{i}"
        cv2.imwrite(os.path.join(out_img_dir, out_stem + ext),
                    cv2.cvtColor(aug["image"], cv2.COLOR_RGB2BGR))
        with open(os.path.join(out_lbl_dir, out_stem + ".txt"), "w") as f:
            for cls, bb in zip(aug["class_labels"], aug["bboxes"]):
                f.write(f"{cls} {bb[0]:.6f} {bb[1]:.6f} {bb[2]:.6f} {bb[3]:.6f}\n")
        saved += 1
    return saved

# ── Identify under-represented classes ───────────────────────────────────────
train_img_dir = os.path.join(CFG.WORKING_PATH, "train", "images")
train_lbl_dir = os.path.join(CFG.WORKING_PATH, "train", "labels")

class_counter: Counter = Counter()
for lbl_file in glob.glob(os.path.join(train_lbl_dir, "*.txt")):
    cls_ids, _ = read_yolo_labels(lbl_file)
    class_counter.update(cls_ids)

print("Training annotation counts per class:")
for cid, name in CFG.PPE_CLASSES.items():
    print(f"   {cid:2d}  {name:<15} {class_counter.get(cid, 0):>5}")

median_count = np.median(list(class_counter.values()))
TARGET_CLASSES_AUG = {cid for cid, cnt in class_counter.items() if cnt < median_count * 0.5}
print(f"\n📌 Classes below 50% of median ({median_count:.0f}) → will augment:")
for cid in sorted(TARGET_CLASSES_AUG):
    print(f"   {cid}  {CFG.PPE_CLASSES[cid]}")

# ── Run augmentation ──────────────────────────────────────────────────────────
total_aug = 0
if TARGET_CLASSES_AUG:
    for lbl_file in glob.glob(os.path.join(train_lbl_dir, "*.txt")):
        cls_ids, _ = read_yolo_labels(lbl_file)
        if any(c in TARGET_CLASSES_AUG for c in cls_ids):
            stem    = Path(lbl_file).stem
            img_src = next((os.path.join(train_img_dir, stem + ext)
                            for ext in EXTS
                            if os.path.exists(os.path.join(train_img_dir, stem + ext))), None)
            if img_src:
                total_aug += augment_image(img_src, lbl_file, train_img_dir, train_lbl_dir, n_copies=2)

print(f"\n✅ Augmented images added : {total_aug}")
print(f"   Training images total  : {len(glob.glob(os.path.join(train_img_dir, '*')))}")

## 🧠 Step 9 — Create Dataset YAML & Train YOLO11

| Variant | Params | mAP50 (COCO) | Recommended for |
|---------|--------|--------------|-----------------|
| yolo11n | ~2.6 M | 39.5 | Edge / fast demo |
| **yolo11s** | ~9.4 M | **47.0** | **Colab T4 ← default** |
| yolo11m | ~20.1 M | 51.5 | Colab A100 |

In [ ]:
# ── Create dataset.yaml ───────────────────────────────────────────────────────
dataset_config = {
    "path":  CFG.WORKING_PATH,
    "train": "train/images",
    "val":   "valid/images",
    "test":  "test/images",
    "nc":    CFG.NUM_CLASSES,
    "names": [CFG.PPE_CLASSES[i] for i in range(CFG.NUM_CLASSES)],
}

with open(CFG.YAML_PATH, "w") as f:
    yaml.dump(dataset_config, f, default_flow_style=False, sort_keys=False)

print(f"✅ YAML saved → {CFG.YAML_PATH}")
print(yaml.dump(dataset_config, default_flow_style=False, sort_keys=False))

In [ ]:
CFG.set_seed()
model = YOLO(CFG.BASE_MODEL)

print(f"🚀 Starting training ─────────────────────────────────")
print(f"   Model    : {CFG.BASE_MODEL}")
print(f"   Epochs   : {CFG.EPOCHS}  (patience={CFG.PATIENCE})")
print(f"   Batch    : {CFG.BATCH_SIZE}  |  imgsz={CFG.IMGSZ}")
print(f"   Device   : {CFG.DEVICE}  |  Optimizer={CFG.OPTIMIZER}")
print(f"   YAML     : {CFG.YAML_PATH}")
print("────────────────────────────────────────────────────────")

train_results = model.train(
    data      = CFG.YAML_PATH,
    epochs    = CFG.EPOCHS,
    patience  = CFG.PATIENCE,
    batch     = CFG.BATCH_SIZE,
    imgsz     = CFG.IMGSZ,
    device    = CFG.DEVICE,
    optimizer = CFG.OPTIMIZER,
    lr0       = CFG.LR0,
    cos_lr    = CFG.COS_LR,
    seed      = CFG.SEED,
    fraction  = CFG.FRACTION,
    rect      = CFG.RECT,
    project   = CFG.WORKING_PATH,
    name      = "ppe_yolo11s",
    exist_ok  = True,
    verbose   = True,
)

print("\n✅ Training complete!")

## 📊 Step 10 — Evaluate Model Performance

In [ ]:
run_dir = os.path.join(CFG.WORKING_PATH, "ppe_yolo11s")

# ── Training curves ───────────────────────────────────────────────────────────
results_csv = os.path.join(run_dir, "results.csv")
if os.path.exists(results_csv):
    df_res = pd.read_csv(results_csv)
    df_res.columns = df_res.columns.str.strip()

    metrics = {
        "Box Loss (train)": ("train/box_loss",       "#e74c3c"),
        "Box Loss (val)":   ("val/box_loss",          "#c0392b"),
        "Cls Loss (train)": ("train/cls_loss",        "#3498db"),
        "Cls Loss (val)":   ("val/cls_loss",          "#2980b9"),
        "mAP50":            ("metrics/mAP50(B)",      "#2ecc71"),
        "mAP50-95":         ("metrics/mAP50-95(B)",   "#27ae60"),
    }

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("YOLO11s Training Curves", fontsize=16, fontweight="bold")
    for ax, pairs in zip(axes, [
        ("Box Loss (train)", "Box Loss (val)"),
        ("Cls Loss (train)", "Cls Loss (val)"),
        ("mAP50", "mAP50-95"),
    ]):
        for label in pairs:
            col, color = metrics[label]
            if col in df_res.columns:
                ax.plot(df_res["epoch"], df_res[col], label=label, color=color, linewidth=2)
        ax.legend(); ax.grid(alpha=0.3); ax.set_xlabel("Epoch")
    axes[0].set_title("Box Loss"); axes[1].set_title("Class Loss"); axes[2].set_title("mAP")
    plt.tight_layout()
    plt.savefig(f"{CFG.WORKING_PATH}/training_curves.png", dpi=120, bbox_inches="tight")
    plt.show()
else:
    print(f"⚠️  results.csv not found at {results_csv}")

In [ ]:
best_weights  = os.path.join(run_dir, "weights", "best.pt")
trained_model = YOLO(best_weights)

val_metrics  = trained_model.val(data=CFG.YAML_PATH, split="val",
                                  imgsz=CFG.IMGSZ, device=CFG.DEVICE, verbose=False)
test_metrics = trained_model.val(data=CFG.YAML_PATH, split="test",
                                  imgsz=CFG.IMGSZ, device=CFG.DEVICE, verbose=False)

def print_metrics(tag, m):
    print(f"\n{'─'*50}\n  {tag}\n{'─'*50}")
    print(f"  Precision : {m.box.mp:.4f}")
    print(f"  Recall    : {m.box.mr:.4f}")
    print(f"  mAP50     : {m.box.map50:.4f}")
    print(f"  mAP50-95  : {m.box.map:.4f}\n")
    rows = []
    for i, name in enumerate(CFG.PPE_CLASSES.values()):
        try:
            rows.append({"Class": name,
                         "P":        round(float(m.box.p[i]),    3),
                         "R":        round(float(m.box.r[i]),    3),
                         "mAP50":    round(float(m.box.ap50[i]), 3),
                         "mAP50-95": round(float(m.box.ap[i]),   3)})
        except IndexError: pass
    print(pd.DataFrame(rows).to_string(index=False))

print_metrics("📋 VALIDATION SET", val_metrics)
print_metrics("📋 TEST SET",        test_metrics)

In [ ]:
# ── Visualise YOLO artefact images ───────────────────────────────────────────
artefact_images = {
    "Confusion Matrix (norm)": os.path.join(run_dir, "confusion_matrix_normalized.png"),
    "Confusion Matrix":        os.path.join(run_dir, "confusion_matrix.png"),
    "F1 Curve":                os.path.join(run_dir, "F1_curve.png"),
    "PR Curve":                os.path.join(run_dir, "PR_curve.png"),
    "P Curve":                 os.path.join(run_dir, "P_curve.png"),
    "R Curve":                 os.path.join(run_dir, "R_curve.png"),
    "Labels":                  os.path.join(run_dir, "labels.jpg"),
}
found = {k: v for k, v in artefact_images.items() if os.path.exists(v)}
if found:
    cols  = 3
    rows  = (len(found) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*7, rows*6))
    axes = axes.flatten()
    for ax, (title, path) in zip(axes, found.items()):
        img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        ax.imshow(img); ax.set_title(title, fontsize=11, fontweight="bold"); ax.axis("off")
    for ax in axes[len(found):]: ax.axis("off")
    plt.tight_layout()
    plt.savefig(f"{CFG.WORKING_PATH}/training_artefacts.png", dpi=120, bbox_inches="tight")
    plt.show()
else:
    print("⚠️  No artefact images found.")

## 🖼️ Step 11 — Inference Visualisation on Test Images

🟢 **Green** → PPE worn correctly  |  🔴 **Red** → PPE missing (violation)  |  ⬜ **Grey** → neutral

In [ ]:
BOX_COLORS = {"positive": (0, 200, 80), "negative": (220, 40, 40), "neutral": (160, 160, 160)}

def box_color(class_name):
    if class_name in CFG.POSITIVE_CLASSES: return BOX_COLORS["positive"]
    if class_name in CFG.NEGATIVE_CLASSES: return BOX_COLORS["negative"]
    return BOX_COLORS["neutral"]

def draw_predictions(image, result):
    img = image.copy()
    compliance = {"wearing": [], "missing": []}
    if result.boxes is None or len(result.boxes) == 0: return img, compliance
    for box in result.boxes:
        cls_id   = int(box.cls[0]); conf = float(box.conf[0])
        cls_name = CFG.PPE_CLASSES.get(cls_id, "unknown")
        color    = box_color(cls_name)
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        label = f"{cls_name} {conf:.2f}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
        cv2.rectangle(img, (x1, y1-th-6), (x1+tw, y1), color, -1)
        cv2.putText(img, label, (x1, y1-4), cv2.FONT_HERSHEY_SIMPLEX, 0.55,
                    (255, 255, 255), 1, cv2.LINE_AA)
        if cls_name in CFG.NEGATIVE_CLASSES: compliance["missing"].append(cls_name)
        elif cls_name in CFG.POSITIVE_CLASSES: compliance["wearing"].append(cls_name)
    return img, compliance

# ── Visualise 8 test images ───────────────────────────────────────────────────
test_img_dir = os.path.join(CFG.WORKING_PATH, "test", "images")
test_imgs    = glob.glob(os.path.join(test_img_dir, "*"))
sample_imgs  = random.sample(test_imgs, min(8, len(test_imgs)))

n_cols = 4; n_rows = (len(sample_imgs) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*5, n_rows*5))
axes = axes.flatten()

for ax, img_path in zip(axes, sample_imgs):
    image   = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    results = trained_model(img_path, verbose=False)
    annotated, comp = draw_predictions(image, results[0])
    status = "✅ Compliant" if not comp["missing"] else f"❌ Missing: {', '.join(set(comp['missing']))}"
    ax.imshow(annotated)
    ax.set_title(f"{Path(img_path).name[:22]}\n{status}", fontsize=8)
    ax.axis("off")
for ax in axes[len(sample_imgs):]: ax.axis("off")

plt.suptitle("Test-Set Predictions  (green=PPE worn, red=PPE missing)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CFG.WORKING_PATH}/test_predictions.png", dpi=120, bbox_inches="tight")
plt.show()

## 💾 Step 12 — Save Model to Google Drive

In [ ]:
# ── Copy best.pt → /content + Google Drive ────────────────────────────────────
shutil.copy(best_weights, CFG.OUTPUT_MODEL_PATH)
shutil.copy(best_weights, CFG.DRIVE_MODEL_PATH)
print(f"✅ Local  → {CFG.OUTPUT_MODEL_PATH}")
print(f"✅ Drive  → {CFG.DRIVE_MODEL_PATH}")

# ── Save metadata JSON ────────────────────────────────────────────────────────
metadata = {
    "model_file":     CFG.OUTPUT_MODEL_PATH,
    "base_model":     CFG.BASE_MODEL,
    "num_classes":    CFG.NUM_CLASSES,
    "class_names":    list(CFG.PPE_CLASSES.values()),
    "imgsz":          CFG.IMGSZ,
    "epochs_trained": CFG.EPOCHS,
    "batch_size":     CFG.BATCH_SIZE,
    "optimizer":      CFG.OPTIMIZER,
    "lr0":            CFG.LR0,
    "cos_lr":         CFG.COS_LR,
    "val_mAP50":      round(float(val_metrics.box.map50),  4),
    "val_mAP50-95":   round(float(val_metrics.box.map),    4),
    "test_mAP50":     round(float(test_metrics.box.map50), 4),
    "test_mAP50-95":  round(float(test_metrics.box.map),   4),
    "train_images":   len(glob.glob(os.path.join(CFG.WORKING_PATH, "train", "images", "*"))),
    "valid_images":   len(glob.glob(os.path.join(CFG.WORKING_PATH, "valid", "images", "*"))),
    "test_images":    len(glob.glob(os.path.join(CFG.WORKING_PATH, "test",  "images", "*"))),
}

with open(CFG.METADATA_PATH,  "w") as fp: json.dump(metadata, fp, indent=4)
with open(CFG.DRIVE_META_PATH,"w") as fp: json.dump(metadata, fp, indent=4)
print(f"\n📄 Metadata saved.")
print(json.dumps(metadata, indent=4))

## 📦 Step 13 — Export Model for Deployment

Export to **ONNX** (universal) for web/backend and **TFLite** (optional) for mobile/edge.  
The ONNX model is saved to Google Drive alongside `best.pt`.

In [ ]:
# ── ONNX export ────────────────────────────────────────────────────────────────
onnx_path = trained_model.export(format="onnx", imgsz=CFG.IMGSZ, dynamic=True)
print(f"✅ ONNX exported → {onnx_path}")

# Copy ONNX to Drive
onnx_drive_path = os.path.join(DRIVE_BASE, "best_ppe_model.onnx")
shutil.copy(onnx_path, onnx_drive_path)
print(f"💾 ONNX saved to Drive → {onnx_drive_path}")

# ── List all saved files ───────────────────────────────────────────────────────
print("\n📂 Files saved to Google Drive:")
for f in os.listdir(DRIVE_BASE):
    size = os.path.getsize(os.path.join(DRIVE_BASE, f)) / 1e6
    print(f"   {f:<35} {size:.1f} MB")

## 🖥️ Step 14 — Interactive Gradio Demo

Run inference directly in the browser with a **public shareable link** (valid for 72 hours).  
- Upload any construction-site image  
- Adjust the confidence slider  
- The compliance report tells you which PPE items are worn / missing

In [ ]:
import gradio as gr

CONF_THRESHOLD = 0.35
IOU_THRESHOLD  = 0.45

_ppe_model = YOLO(CFG.OUTPUT_MODEL_PATH)

def predict_ppe(image: np.ndarray, conf: float = CONF_THRESHOLD):
    if image is None:
        return None, "⚠️  No image provided."
    results  = _ppe_model(image, conf=conf, iou=IOU_THRESHOLD, verbose=False)
    annotated, compliance = draw_predictions(image, results[0])
    wearing_set = set(compliance["wearing"])
    missing_set = set(compliance["missing"])
    lines = ["## PPE Compliance Report\n"]
    if wearing_set:
        lines.append("### ✅ PPE Detected (worn correctly)")
        for item in sorted(wearing_set): lines.append(f"- {item}")
        lines.append("")
    if missing_set:
        lines.append("### ❌ Safety Violations (PPE missing)")
        for item in sorted(missing_set): lines.append(f"- {item}")
        lines.append("")
    overall = ("🟢 **COMPLIANT** — All critical PPE present."
               if not missing_set
               else "🔴 **NON-COMPLIANT** — Immediate action required.")
    lines.append(f"\n**Overall Status:** {overall}")
    return annotated, "\n".join(lines)

# Example images from test set
example_images = random.sample(
    glob.glob(os.path.join(CFG.WORKING_PATH, "test", "images", "*")),
    min(4, len(glob.glob(os.path.join(CFG.WORKING_PATH, "test", "images", "*"))))
)

with gr.Blocks(title="PPE Kit Detection", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🦺 PPE Kit Detection — Construction Site Safety
    Powered by **YOLO11s** trained on the Ketaki PPE dataset.  
    Upload a construction-site image and click **Detect PPE**.
    """)
    with gr.Row():
        with gr.Column(scale=1):
            input_image = gr.Image(label="Input Image", type="numpy", height=480)
            conf_slider = gr.Slider(0.1, 0.9, value=CONF_THRESHOLD, step=0.05,
                                    label="Confidence Threshold")
            detect_btn  = gr.Button("🔍 Detect PPE", variant="primary")
        with gr.Column(scale=1):
            output_image  = gr.Image(label="Annotated Output", height=480)
            output_report = gr.Markdown(label="Compliance Report")

    detect_btn.click(fn=predict_ppe,
                     inputs=[input_image, conf_slider],
                     outputs=[output_image, output_report])

    if example_images:
        gr.Examples(
            examples=[[p, CONF_THRESHOLD] for p in example_images],
            inputs=[input_image, conf_slider],
            outputs=[output_image, output_report],
            fn=predict_ppe,
            cache_examples=False,
            label="Example Images from Test Set",
        )

    gr.Markdown("""
    ---
    **Classes detected:**  
    🟢 *Helmet · Gloves · Vest · Boots · Goggles · Person*  
    🔴 *no_helmet · no_goggle · no_gloves · no_boots*
    """)

demo.launch(share=True, debug=False)

## 🏁 Conclusion & Next Steps

### What We Built
An end-to-end **YOLO11s** PPE detection pipeline trained on the Ketaki Construction Site dataset, optimised for **Google Colab T4 GPU**.

### Pipeline Summary

| Step | Details |
|------|---------|
| Dataset | Ketaki PPE — 11 classes (6 positive, 4 negative, 1 neutral) |
| Data setup | Kaggle API download → stratified 75/15/10 re-split → pHash dedup |
| Augmentation | HFlip · Brightness/Contrast · HSV · GaussNoise for rare classes |
| Model | **YOLO11s** — AdamW · cosine LR · 50 epochs · batch 32 |
| Evaluation | Per-class P/R/mAP50/mAP50-95 on val & test splits |
| Saved outputs | `best.pt` + `best.onnx` + `model_metadata.json` → **Google Drive** |
| Interface | Gradio app with live confidence slider + compliance report |

### 🚀 Next Steps — Frontend
After training, use the saved `best.pt` / `best_ppe_model.onnx` to build a frontend:
1. **Flask / FastAPI** — `/predict` endpoint that accepts image uploads and returns JSON detections
2. **Streamlit** — quick interactive dashboard with file uploader
3. **React + ONNX Runtime Web** — fully in-browser inference (no server needed)

The model metadata JSON contains all you need to wire up the frontend (class names, imgsz, thresholds).

---
*Notebook trained on Google Colab — part of a Kaggle Computer Vision portfolio.*